In [1]:
import numpy as np

In [2]:
class CircularBuffer:
    """Circular buffer for complex IQ samples with protection.

    Three logical pointers tracked as ABSOLUTE positions (monotonically
    increasing, never wrap). Physical positions computed via modulo.
    """

    def __init__(self, buffer_size):
        self.buffer_size = buffer_size
        self.buffer = np.empty(buffer_size, dtype=complex)

        self._abs_write = 0
        self._abs_read = 0
        self._abs_protect = 0

        self._protections = {}   
        self._next_protection_id = 0
    
    def _physical(self, abs_pos):
        return abs_pos % self.buffer_size
    
    @property
    def available(self):
        return self._abs_write - self._abs_read

    @property
    def free_space(self):
        if self._protections:
            oldest = min(self._abs_protect, self._abs_read)
        else:
            oldest = self._abs_read
        return self.buffer_size - (self._abs_write - oldest)
    

    # ---- write side (called by ingestion thread) ----

    def write(self, data):
        n = len(data)
        if n > self.free_space:
            return False
        # get physical position in the buffer
        phys = self._physical(self._abs_write)
        first_part = min(n, self.buffer_size - phys)
        # before boundary
        self.buffer[phys:phys + first_part] = data[:first_part]
        # if after boundary
        if first_part < n:
            self.buffer[:n - first_part] = data[first_part:]
        # absolute position
        self._abs_write += n
        return True
    
    # ---- read side ----

    def read(self, offset, length):
        if offset + length > self.available:
            return None
        phys = self._physical(self._abs_read + offset)
        return self._read_physical(phys, length)

    def _read_physical(self, phys_start, length):
        first_part = min(length, self.buffer_size - phys_start)
        # before boundary
        result = self.buffer[phys_start:phys_start + first_part]
        # if after boundary
        if first_part < length:
            result = np.concatenate((result, self.buffer[:length - first_part]))
        return result
    
    def consume(self, length):
        """Advance _abs_read by length. No check protections."""
        if length > self.available:
            return False
        self._abs_read += length
 
        if not self._protections:
            self._abs_protect = self._abs_read
        return True
    
    # ---- protection side (eager) ----

    def protect(self, offset, length):
        """Mark a region as protected. Return protection id (pid)"""
        abs_start = self._abs_read + offset
        pid = self._next_protection_id
        self._next_protection_id += 1
 
        self._protections[pid] = {
            'abs_start': abs_start,
            'length': length
        }
 
        self._update_abs_protect()
        return pid
    
    def read_protected(self, pid):
        """Read data using a pid."""
        prot = self._protections.get(pid)
        if prot is None:
            print(f'WARNING: unknown protection ID {pid}')
            return None
 
        phys_start = self._physical(prot['abs_start'])
        return self._read_physical(phys_start, prot['length'])

    def _update_abs_protect(self):
        """Recompute abs_protect as the minimum of all active protections."""
        if self._protections:
            self._abs_protect = min(
                p['abs_start'] for p in self._protections.values())
        else:
            self._abs_protect = self._abs_read
    
    def release(self, pid):
        """Release a protection. Return True if released."""
        if pid not in self._protections:
            print(f'WARNING: unknown protection ID {pid}')
            return False
 
        del self._protections[pid]
        self._update_abs_protect()
        return True

    
    def reset(self):
        self.buffer.fill(0)
        self._abs_write = 0
        self._abs_read = 0
        self._abs_protect = 0
        self._protections.clear()
        self._next_protection_id = 0

### Test
1. free_space

In [3]:
buf = CircularBuffer(100)

# Case 1: _write=60, _read=20, no protection
buf._abs_write = 60
buf._abs_read = 20
buf._abs_protect = 0
buf._protections = {}

assert buf.free_space == 60

# Case 2: _write=120, _read=60, no protection
buf._abs_write = 120
buf._abs_read = 60
buf._abs_protect = 0
buf._protections = {}

assert buf.free_space == 40

# Case 3: _write=60, _read=20, protect=40
buf._abs_write = 60
buf._abs_read = 20
buf._abs_protect = 40
buf._protections = {0: True}

assert buf.free_space == 60

# Case 4: _write=110, _read=20, protect=40
buf._abs_write = 110
buf._abs_read = 20
buf._abs_protect = 40
buf._protections = {0: True}

assert buf.free_space == 10

# Case 5: _write=110, _read=20, protect=105
buf._abs_write = 110
buf._abs_read = 20
buf._abs_protect = 105
buf._protections = {0: True}

assert buf.free_space == 10

2. write

In [4]:
buf = CircularBuffer(100)

# Case 1: write 60
data = np.arange(60, dtype=complex)
result = buf.write(data)

assert buf._abs_write == 60

# Case 2: _read=20, _write=40, write 70
buf.reset()
buf._abs_read = 20
buf._abs_write = 40
data = np.arange(70, dtype=complex)
result = buf.write(data)

assert buf._abs_write == 110

# Case 3: _read=20, _write=40, write 90
buf.reset()
buf._abs_read = 20
buf._abs_write = 40
data = np.arange(90, dtype=complex)
result = buf.write(data)

assert buf._abs_write == 40
assert not result

3. read and _read_physical

In [5]:
buf = CircularBuffer(100)

# Case 1: write 60, read 40
data = np.arange(60, dtype=complex)
_ = buf.write(data)
result = buf.read(0,40)

assert np.any(result == np.arange(40, dtype=complex))

# Case 2: write 60, read 80
buf.reset()
data = np.arange(60, dtype=complex)
_ = buf.write(data)
result = buf.read(0,80)

assert not result

# Case 3: write 60, offset 20, read 40
buf.reset()
data = np.arange(60, dtype=complex)
_ = buf.write(data)
result = buf.read(20,40)

assert np.any(result == np.arange(20,60, dtype=complex))


# Case 3: _write=80, _read=60, write 60, offset 20, read 40
buf.reset()
buf._abs_write = 80
buf._abs_read = 60
data = np.arange(60, dtype=complex)
_ = buf.write(data)
result = buf.read(20,40)

assert np.any(result == np.arange(40, dtype=complex))

4. consume

In [6]:
buf = CircularBuffer(100)

# Case 1: _write=80, _read=60, write 60, consume 20, offset 20, read 40
buf._abs_write = 80
buf._abs_read = 60
data = np.arange(60, dtype=complex)
_ = buf.write(data)
buf.consume(20)
result = buf.read(20,40)

assert np.any(result == np.arange(20,60, dtype=complex))
assert buf._abs_protect == 80

# Case 2: _write=80, _read=60, write 60, consume 20, offset 20, read 40
buf._abs_write = 80
buf._abs_read = 60
data = np.arange(60, dtype=complex)
_ = buf.write(data)
result = buf.read(20,40)
buf.consume(20)

assert np.any(result == np.arange(40, dtype=complex))
assert buf._abs_read == 80

# Case 3: _write=80, _read=60, _protect=70, write 60, consume 20, offset 20, read 40
buf.reset()
buf._abs_write = 80
buf._abs_read = 60
buf._abs_protect = 70
buf._protections = {0: True}
data = np.arange(60, dtype=complex)
_ = buf.write(data)
buf.consume(20)
result = buf.read(20,40)

assert np.any(result == np.arange(20,60, dtype=complex))
assert buf._abs_protect == 70

5. protection API

In [7]:
buf = CircularBuffer(100)

# Case 1: write 80, protect 20, offset 20, consume 60
data = np.arange(80, dtype=complex)
_ = buf.write(data)
pid = buf.protect(20,20)
buf.consume(60)
result = buf.read_protected(pid)

assert np.any(result == np.arange(20,40, dtype=complex))
assert buf._abs_write == 80
assert buf._abs_read == 60
assert buf._abs_protect == 20

# Case 2: write 80, protect 20, offset 20, consume 60, release (20,20)
buf.reset()
data = np.arange(80, dtype=complex)
_ = buf.write(data)
pid = buf.protect(20,20)
buf.consume(60)
result = buf.read_protected(pid)
buf.release(pid)

assert np.any(result == np.arange(20,40, dtype=complex))
assert buf._abs_write == 80
assert buf._abs_read == 60
assert buf._abs_protect == 60

# Case 2: write 80, protect 20, offset 20, consume 60, write 30, protect 10, offset 40, release (20,20)
buf.reset()
data = np.arange(80, dtype=complex)
_ = buf.write(data)
pid1 = buf.protect(20,20)
buf.consume(60)
data = np.arange(80,110, dtype=complex)
_ = buf.write(data)
pid2 = buf.protect(40,10)
result = buf.read_protected(pid2)

assert buf._abs_protect == 20
buf.release(pid1)

assert np.any(result == np.arange(100,110, dtype=complex))
assert buf._abs_write == 110
assert buf._abs_read == 60
assert buf._abs_protect == 100

buf.release(pid2)
assert buf._abs_protect == 60